# IntelliSum: results analysis

Reads the per-article results written by `experiments/scripts/run_comparison.py` and explores them beyond the summary tables produced by `summarize_results.py`:

1. Overall ROUGE per method (on the articles all methods completed)
2. CNN vs Daily Mail
3. Summary length vs ROUGE
4. Speed
5. Faithfulness flags (BART / Hybrid)
6. Example summaries side by side

Run from the repository root with the backend virtual environment (`pip install -r experiments/requirements.txt jupyter`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path('experiments/results/test_short_seed42') if Path('experiments').exists() else Path('../results/test_short_seed42')
df = pd.read_csv(RESULTS / 'per_article.csv')
ok = df[df.status == 'ok'].copy()
for col in [c for c in ok.columns if c.endswith('_f')]:
    ok[col] = 100 * ok[col]  # report ROUGE on the usual 0-100 scale

ORDER = [m for m in ['lead3', 'tfidf', 'textrank', 'bart', 'hybrid'] if m in set(ok.method)]
BLUE = '#2a78d6'
print(ok.groupby('method').id.nunique().reindex(ORDER))

## 1. ROUGE on the articles every method completed

Comparing means is only fair on identical articles, so restrict to the articles all methods finished.

In [ ]:
complete = ok.groupby('id').method.nunique()
common = ok[ok.id.isin(complete[complete == len(ORDER)].index)]
table = common.groupby('method')[['rouge1_f', 'rouge2_f', 'rougeL_f', 'rougeLsum_f', 'summary_words', 'seconds']].mean()
table.reindex(ORDER).round(2)

## 2. CNN vs Daily Mail

The two sources behave differently (Daily Mail highlights overlap more with the lead sentences), which is why the experiments sample randomly across the whole split.

In [ ]:
common.pivot_table(index='method', columns='source', values='rouge1_f', aggfunc='mean').reindex(ORDER).round(2)

## 3. Summary length vs ROUGE

ROUGE F1 depends on length: longer summaries gain recall and lose precision. Methods that produce summaries of very different lengths are therefore not perfectly comparable; the reference highlights are much shorter than a 12 % summary.

In [ ]:
print('mean reference length (words):', round(common.reference_words.mean(), 1))
print(common.groupby('method')[['summary_words', 'rouge1_p', 'rouge1_r']].mean().reindex(ORDER).round(3))

fig, axes = plt.subplots(1, len(ORDER), figsize=(3 * len(ORDER), 2.8), sharey=True)
for ax, method in zip(axes, ORDER):
    sub = common[common.method == method]
    ax.scatter(sub.summary_words, sub.rouge1_f, s=10, color=BLUE, alpha=0.6)
    ax.set_title(method)
    ax.set_xlabel('summary words')
axes[0].set_ylabel('ROUGE-1 F1')
fig.tight_layout()

## 4. Speed

In [ ]:
ok.groupby('method').seconds.describe(percentiles=[0.5, 0.9]).reindex(ORDER).round(3)

## 5. Faithfulness flags (experimental heuristic)

Share of BART / Hybrid summaries with at least one sentence flagged as potentially unsupported. A flag is a prompt to check, not proof of an error.

In [ ]:
abstractive = ok[ok.method.isin(['bart', 'hybrid'])].copy()
abstractive['flagged'] = pd.to_numeric(abstractive.flagged_sentences, errors='coerce').fillna(0) > 0
abstractive.groupby('method').flagged.agg(['sum', 'mean', 'count'])

## 6. Example summaries

Qualitative comparison on one article (change `example` to browse).

In [ ]:
example = common.id.unique()[0]
for _, row in common[common.id == example].set_index('method').reindex(ORDER).iterrows():
    print(f"--- {row.name} (ROUGE-1 {row.rouge1_f:.1f}, {row.summary_words:.0f} words)\n{row.summary}\n")